# Statistics Villains: Web Scraping with pandas

**Big idea:** web scraping means using code to retrieve information from a webpage. In this lesson, we retrieve an HTML table, turn it into a pandas DataFrame, clean it, and answer simple questions.

The examples use the instructor's **Statistics Villains** page. The default run uses a six-row teaching sample, so the notebook also works when you are offline. The villains and costs are fictional.

**Understand → Predict → Run → Change → Use → Practice → Connect**

[View on GitHub](https://github.com/mathguy-uf/MAD2502/blob/main/applied_data/03_statistics_villains.ipynb) · [Run in Google Colab](https://colab.research.google.com/github/mathguy-uf/MAD2502/blob/main/applied_data/03_statistics_villains.ipynb) · [Launch Binder/JupyterLab](https://mybinder.org/v2/gh/mathguy-uf/MAD2502/main?urlpath=lab/tree/applied_data/03_statistics_villains.ipynb)

Run cells from the top with **Shift+Enter**. If something becomes confusing, restart the kernel and run all cells again in order.

In [ ]:
# If an import fails, uncomment and run this once, then restart the kernel.
# %pip install pandas requests lxml

In [ ]:
from io import StringIO
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from IPython.display import display

## Understand — webpage, HTML, and web scraping

A **webpage** is usually sent to your browser as **HTML** (HyperText Markup Language). HTML uses tags to describe the parts of a page.

A few table tags are:

- `<table>` — a table
- `<tr>` — a table row
- `<th>` — a heading cell
- `<td>` — a data cell

**Web scraping** means using code to retrieve information from a webpage instead of copying it by hand. Scraping is useful when a page contains data but does not provide a better machine-readable source.

Before scraping, look for a **CSV download** or an **API**. Those are usually preferable because they are designed for data access. HTML is designed mainly for displaying a webpage, so a redesign can break scraping code.

### `robots.txt`

Many websites publish a file named `robots.txt`. It gives instructions to automated programs about which parts of a site they may request.

`robots.txt` is **not** the same as permission to reuse data, and it is not a guarantee that scraping is allowed. Also follow the site's terms and use a reasonable request rate.

In this lesson, live scraping is optional. The notebook checks `robots.txt` before requesting the teaching page.

## Understand — requests, responses, and status codes

`requests.get(url)` sends an HTTP **GET request** to a URL. The server sends back a **response**.

Useful response information includes:

- `response.status_code` — a number describing what happened
- `response.text` — the returned text, such as HTML

Common status codes:

- **200** — success
- **403** — forbidden
- **404** — page not found
- **429** — too many requests

`response.raise_for_status()` stops with an exception for unsuccessful HTTP status codes. A `timeout` prevents the program from waiting forever.

In [ ]:
PAGE_URL = "https://people.clas.ufl.edu/mathguy/supervillain-statistics-playground/"
ROBOTS_URL = "https://people.clas.ufl.edu/robots.txt"
USER_AGENT = "MAD2502Teaching/1.0"

# Keep this False for the normal lesson.
# Change it to True only when you want to try the live webpage.
RUN_LIVE = False

SAMPLE_HTML = """
<table>
  <thead>
    <tr>
      <th>Villain Name</th>
      <th>Insurance Cost</th>
      <th>Number of Henchmen</th>
      <th>Villain Category</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Condiment King</td><td>$7,900</td><td>2</td><td>Human</td></tr>
    <tr><td>Calendar Man</td><td>$2,500</td><td>3</td><td>Human</td></tr>
    <tr><td>Polka-Dot Man</td><td></td><td></td><td>Human</td></tr>
    <tr><td>Kite Man</td><td></td><td></td><td>Human</td></tr>
    <tr><td>Crazy Quilt</td><td>$9,600</td><td>1</td><td>Human</td></tr>
    <tr><td>Orca</td><td>$9,100</td><td>1</td><td>Mutate</td></tr>
  </tbody>
</table>
"""

html = SAMPLE_HTML
source_label = "six-row teaching sample"

## Predict — what does a request return?

Before running the next cell, predict:

1. Does `requests.get()` return a DataFrame?
2. If the request succeeds, what Python type do you expect `response.text` to have?
3. If a webpage contains three HTML tables, how many tables might `pd.read_html()` find?

In [ ]:
if RUN_LIVE:
    try:
        robots_response = requests.get(
            ROBOTS_URL,
            headers={"User-Agent": USER_AGENT},
            timeout=30,
        )
        print("robots.txt status:", robots_response.status_code)
        robots_response.raise_for_status()

        policy = RobotFileParser()
        policy.parse(robots_response.text.splitlines())

        if not policy.can_fetch(USER_AGENT, PAGE_URL):
            raise ValueError("robots.txt does not allow this request.")

        response = requests.get(
            PAGE_URL,
            headers={"User-Agent": USER_AGENT},
            timeout=30,
        )
        print("page status:", response.status_code)
        response.raise_for_status()

        html = response.text
        source_label = "live instructor webpage"

    except (requests.RequestException, ValueError) as error:
        print("Live retrieval unavailable:", error)
        print("Using the teaching sample instead.")

print("Source:", source_label)
print("HTML object type:", type(html))
print("First 120 characters:")
print(html[:120])

## Run — let pandas find HTML tables

`pd.read_html(...)` looks through HTML and returns a **Python list of DataFrames**.

Why a list? A webpage can contain **zero, one, or many tables**. Returning a list gives pandas a place to put every table it finds.

Lists use positions starting at 0:

- `tables[0]` is the first table
- `tables[1]` is the second table

`StringIO(html)` lets pandas treat the HTML text like a small file.

In [ ]:
tables = pd.read_html(StringIO(html))

print("Number of tables found:", len(tables))
print("Object type:", type(tables))

for index, table in enumerate(tables):
    print("\nTable", index)
    print("shape:", table.shape)
    print("columns:", list(table.columns))
    display(table.head())

### Select the table you actually want

Do not automatically assume `tables[0]` is the correct table on every website. A page may contain menus, schedules, or other tables.

Here we look for a table containing the three column headings we expect. `issubset(...)` checks whether all required headings are present.

In [ ]:
required_columns = {
    "Villain Name",
    "Insurance Cost",
    "Number of Henchmen",
}

matches = [table for table in tables if required_columns.issubset(table.columns)]

if len(matches) != 1:
    raise ValueError(
        "Expected exactly one villain table. Inspect the tables before continuing."
    )

villains = matches[0].copy()
display(villains.head())

## Change — clean column names

Column names often arrive with spaces, capitalization, punctuation, or other awkward characters. We will make these names easier to type by:

1. removing spaces at the beginning and end,
2. converting to lowercase,
3. replacing spaces with underscores.

**Predict:** what will `Insurance Cost` become?

In [ ]:
villains.columns = [
    str(name).strip().lower().replace(" ", "_") for name in villains.columns
]

print(list(villains.columns))
assert villains.columns.is_unique

## Change — convert text to numbers

A value such as `"$7,900"` is text, not a number. Python cannot correctly average it until the `$` and comma are removed.

We first preserve the text, then remove those characters. `pd.to_numeric(..., errors="coerce")` converts valid values to numbers. Anything it cannot convert becomes a missing value (`NaN`).

Missing values mean **unknown or unavailable**, not automatically zero.

In [ ]:
cost_text = villains["insurance_cost"].astype("string")
cost_text = cost_text.str.replace("$", "", regex=False)
cost_text = cost_text.str.replace(",", "", regex=False)

villains["insurance_cost"] = pd.to_numeric(
    cost_text,
    errors="coerce",
)
villains["number_of_henchmen"] = pd.to_numeric(
    villains["number_of_henchmen"],
    errors="coerce",
)

print(villains.dtypes)
print("\nMissing values by column:")
print(villains.isna().sum())
display(villains)

## Use — filter, sort, and summarize

A **filter** keeps only rows that satisfy a condition.

The expression `villains["insurance_cost"] >= 8000` produces `True` or `False` for each row. Using that expression inside `villains[...]` keeps the rows where the result is `True`.

`sort_values(...)` changes the order of the rows. `mean()` and `median()` provide simple summaries of the known numeric values.

In [ ]:
expensive = villains[villains["insurance_cost"] >= 8000]
expensive = expensive.sort_values(
    "insurance_cost",
    ascending=False,
)

display(expensive)

known_costs = villains["insurance_cost"].count()
print("Known costs:", known_costs, "of", len(villains))
print("Mean known cost:", villains["insurance_cost"].mean())
print("Median known cost:", villains["insurance_cost"].median())

## Practice

1. Change the filter from 8000 to 9000. Predict the remaining villains before running the cell.
2. Sort the complete table by `number_of_henchmen`.
3. Count the missing values in `number_of_henchmen`.
4. Find the mean number of henchmen using only known values.
5. Explain why replacing every missing value with 0 could give a misleading result.

## Connect — when should you scrape?

Prefer a **CSV download** or an **API** when one is available.

- A CSV is already organized as data.
- An API is designed for programs to request data and often returns JSON.
- HTML is designed mainly for webpage display and can change when the page is redesigned.
- Scraping therefore needs extra checks: access rules, status codes, table selection, and cleaning.

For a project, record the page URL, access date, what one row represents, which table you selected, and every cleaning decision you made.